https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/cohorts/2026/06-batch/homework.md

For this homework we will be using the Yellow 2025-11 data from the official website:

In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-13 17:08:12--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 99.84.149.182, 99.84.149.73, 99.84.149.117, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|99.84.149.182|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  13.9MB/s    in 4.7s    

2026-03-13 17:08:17 (14.5 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [1]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as t

### Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

In [2]:
spark = SparkSession.builder \
            .master('local[*]') \
            .appName('homework') \
            .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 14:12:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

Ans: v4.1.1

### Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [8]:
df_trip_data = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df_trip_data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [9]:
df_trip_data.repartition(4).write.parquet('data/yellow/2025/11')

In [12]:
!ls -lah data/yellow/2025/11 

total 99M
drwxr-xr-x 2 gradient gradient 4.0K Mar 14 14:18 .
drwxr-xr-x 3 gradient gradient 4.0K Mar 14 14:18 ..
-rw-r--r-- 1 gradient gradient    8 Mar 14 14:18 ._SUCCESS.crc
-rw-r--r-- 1 gradient gradient 196K Mar 14 14:18 .part-00000-32d6510f-7662-402f-b73e-6496038bd8da-c000.snappy.parquet.crc
-rw-r--r-- 1 gradient gradient 196K Mar 14 14:18 .part-00001-32d6510f-7662-402f-b73e-6496038bd8da-c000.snappy.parquet.crc
-rw-r--r-- 1 gradient gradient 196K Mar 14 14:18 .part-00002-32d6510f-7662-402f-b73e-6496038bd8da-c000.snappy.parquet.crc
-rw-r--r-- 1 gradient gradient 196K Mar 14 14:18 .part-00003-32d6510f-7662-402f-b73e-6496038bd8da-c000.snappy.parquet.crc
-rw-r--r-- 1 gradient gradient    0 Mar 14 14:18 _SUCCESS
-rw-r--r-- 1 gradient gradient  25M Mar 14 14:18 part-00000-32d6510f-7662-402f-b73e-6496038bd8da-c000.snappy.parquet
-rw-r--r-- 1 gradient gradient  25M Mar 14 14:18 part-00001-32d6510f-7662-402f-b73e-6496038bd8da-c000.snappy.parquet
-rw-r--r-- 1 gradient gradient  25M Mar 14 1

Ans: 25 Mb

### Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

In [14]:
df_trip_data.createOrReplaceTempView('trip_data')

In [16]:
spark.sql("""
SELECT 
    COUNT(1) 
FROM 
    trip_data
WHERE
    date_trunc('day', tpep_pickup_datetime) = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [19]:
spark.sql("""
SELECT 
    date_trunc('day', tpep_pickup_datetime) AS date,
    COUNT(1) AS trips_count
FROM 
    trip_data
GROUP BY
    1
ORDER BY
    date
""").show()

+-------------------+-----------+
|               date|trips_count|
+-------------------+-----------+
|2008-12-31 00:00:00|          1|
|2009-01-01 00:00:00|          2|
|2025-10-31 00:00:00|         21|
|2025-11-01 00:00:00|     183032|
|2025-11-02 00:00:00|     152130|
|2025-11-03 00:00:00|     129816|
|2025-11-04 00:00:00|     131512|
|2025-11-05 00:00:00|     142587|
|2025-11-06 00:00:00|     158170|
|2025-11-07 00:00:00|     147795|
|2025-11-08 00:00:00|     153432|
|2025-11-09 00:00:00|     133197|
|2025-11-10 00:00:00|     131860|
|2025-11-11 00:00:00|     131252|
|2025-11-12 00:00:00|     142267|
|2025-11-13 00:00:00|     156277|
|2025-11-14 00:00:00|     154779|
|2025-11-15 00:00:00|     162604|
|2025-11-16 00:00:00|     136369|
|2025-11-17 00:00:00|     130552|
+-------------------+-----------+
only showing top 20 rows


Ans: 162,604

### Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

In [29]:
spark.sql("""
SELECT 
    TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) / 60 AS duration
FROM 
    trip_data
ORDER BY
    duration DESC
LIMIT 
    5
""").show()

+-----------------+
|         duration|
+-----------------+
|90.63333333333334|
|76.93333333333334|
|             76.2|
|69.28333333333333|
|67.06666666666666|
+-----------------+



Ans: 90.6

### Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

Ans: 4040

### Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

In [31]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-14 14:30:21--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 99.84.149.165, 99.84.149.182, 99.84.149.117, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|99.84.149.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-14 14:30:21 (210 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [36]:
df_taxi_zone = spark.read.csv('taxi_zone_lookup.csv', header=True)
df_taxi_zone.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [37]:
df_taxi_zone.createOrReplaceTempView('taxi_zone')

In [43]:
spark.sql("""
SELECT 
    taxi_zone.Zone,
    COUNT(1) AS pickup_frequent
FROM trip_data
LEFT JOIN 
    taxi_zone
    ON trip_data.PULocationID = taxi_zone.LocationID
GROUP BY
    1
ORDER BY 
    pickup_frequent   
""").show(5)

+--------------------+---------------+
|                Zone|pickup_frequent|
+--------------------+---------------+
|Governor's Island...|              1|
|Eltingville/Annad...|              1|
|       Arden Heights|              1|
|       Port Richmond|              3|
|       Rikers Island|              4|
+--------------------+---------------+
only showing top 5 rows


Ans: Arden Heights